<div align="center">

# 🏆 WC 2026 Match Predictor
## XGBoost + Poisson Regression + Live Dashboard

*Predicting the 2026 FIFA World Cup — One Match at a Time*

---

| | |
|:---|:---|
| 👤 **Author** | [Tariq Elnaggar](https://www.kaggle.com/tarekelnaggar) |
| 🐙 **GitHub** | [tito644/wc2026-match-predictor](https://github.com/tito644/wc2026-match-predictor) |
| 💼 **LinkedIn** | [tarek-mohamed-el-naggar](https://www.linkedin.com/in/tarek-mohamed-el-naggar/) |
| 🏆 **Tournament** | FIFA World Cup 2026 — 48 Teams · 104 Matches · 3 Host Nations |
| 🤖 **Models** | XGBoost Classifier + Poisson Regression (Ensemble) |
| 📊 **Training Data** | 49,477 international matches (1872–2026) |
| ⚡ **Live Updates** | Auto-refreshes hourly from GitHub |

</div>

---

> **If you find this useful, please give it an upvote ⬆️ — it helps others discover the work!**


## 📋 Table of Contents

1. [Project Overview & Key Features](#1)
2. [Data Sources](#2)
3. [Setup & Imports](#3)
4. [Data Collection](#4)
5. [Team Name Normalization](#5)
6. [Data Cleaning & Tournament Weighting](#6)
7. [ELO Rating Calculation](#7)
8. [Penalty Shootout Statistics](#8)
9. [Feature Engineering — 22 Features](#9)
10. [Building the Training Dataset](#10)
11. [Live Tournament Update Engine](#11)
12. [Model Training — XGBoost](#12)
13. [Model Training — Poisson Regression](#13)
14. [Feature Importance Analysis](#14)
15. [Match Prediction Function](#15)
16. [Monte Carlo Championship Simulation](#16)
17. [Results & Sample Predictions](#17)
18. [Dashboard Overview](#18)


<a id="1"></a>
## 1. 🎯 Project Overview & Key Features

The **2026 FIFA World Cup** is the first-ever 48-team edition hosted across the USA, Canada, and Mexico.
This notebook builds a **complete end-to-end ML pipeline** that answers one question:

> *Who will win — and by how much?*

---

### 🧠 What Makes This Different

Most football prediction models rely solely on FIFA rankings or basic ELO.
This project introduces **9 categories of features** — including several rarely used in the literature:

| Feature | Description | Novel? |
|---------|-------------|--------|
| ⚡ **Live ELO Rating** | Dynamic strength score — recalculated after every match | |
| 📈 **Recent Form** | Win rate, goals scored/conceded over last 5 & 10 official matches | |
| ⚔️ **Head-to-Head** | Historical win rate + World Cup-specific H2H records | |
| 👥 **Squad Strength** | Average EA FC 26 Overall Rating of top 15 players per squad | ⭐ |
| 💰 **Market Value** | Total Transfermarkt squad valuation in Euros | |
| 🎯 **Penalty Shootouts** | Historical penalty win rate — the "clutch factor" | ⭐ |
| 🔥 **Knockout Pressure** | Binary flag: is this a must-win match? | ⭐ |
| 🏆 **International Caps** | Average experience level per squad | |
| ⚠️ **Upset Potential** | Detects when a weaker team is in significantly better form | ⭐ |

> ⭐ = Features not commonly found in other prediction models

---

### 🏗️ Pipeline Architecture

```
📥 Data Collection     →  49,477 matches from GitHub + 4 Kaggle datasets
🔧 Feature Engineering →  22 features across 9 categories
⚡ Live Update Engine   →  Hourly refresh of tournament state
🤖 Model Training      →  XGBoost Classifier + Poisson Regression (Ensemble)
🖥️  Dashboard          →  6-page Streamlit app with PDF export
```

---

### 🎯 The ELO Dominance Problem (and How We Solved It)

Early versions of the model suffered from **ELO Dominance** — the model defaulted to
"the stronger team always wins," ignoring form, context, and the beautiful chaos of football.

**The fix:** Convert raw ELO difference to a calibrated **win probability**:

$$P(win) = \frac{1}{1 + 10^{-\Delta ELO / 400}}$$

This means even a **+300 ELO advantage only yields ~88% win probability** — not 100%.
Combined with a ±200 ELO cap and three new upset features, the model became significantly more balanced.


<a id="2"></a>
## 2. 📊 Data Sources

| # | Dataset | Source | Records | Content |
|---|---------|--------|---------|---------|
| DS-1 | International Football Results | [martj42/GitHub](https://github.com/martj42/international_results) | **49,477 matches** | Complete history 1872–2026, updates daily |
| DS-2 | FIFA World Rankings | [Kaggle](https://www.kaggle.com/datasets/cashncarry/fifaworldranking) | 1992–2024 | Monthly official FIFA rankings |
| DS-3 | WC 2026 Live Dataset | [Kaggle](https://www.kaggle.com/datasets/mominullptr/fifa-world-cup-2026-dataset) | 1,248 players | Squad data, market values, live match stats |
| DS-4 | EAFC 26 Player Database | [Kaggle](https://www.kaggle.com/datasets/flynn28/eafc26-player-database) | 16,228 players | EA Sports FC 26 player ratings |
| DS-5 | Transfermarkt Dataset | [Kaggle](https://www.kaggle.com/datasets/xfkzujqjvx97n/football-datasets/) | 5.7M+ records | Market values, injury history |

> **DS-1 (results + shootouts) is downloaded automatically from GitHub — no manual setup needed.**
> **DS-2 through DS-5 should be added as Kaggle input datasets.**


<a id="3"></a>
## 3. ⚙️ Setup & Imports


In [1]:
# ============================================================
# SETUP & IMPORTS
# Install any missing packages and load all required libraries
# ============================================================

# Install required packages (in case they're not available)
import subprocess
subprocess.run(["pip", "install", "xgboost", "scikit-learn", "scipy", "-q"], 
               capture_output=True)

# Core libraries
import pandas as pd
import numpy as np
import warnings
import os
import random
from datetime import datetime

# Machine Learning
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

# Statistics
from scipy.stats import poisson as sp_poisson

warnings.filterwarnings("ignore")

print("=" * 55)
print("  WC 2026 Match Predictor — Environment Ready")
print("=" * 55)
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print(f"  xgboost : {xgb.__version__}")
print(f"  Run date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 55)


  WC 2026 Match Predictor — Environment Ready
  pandas  : 2.3.3
  numpy   : 2.0.2
  xgboost : 3.2.0
  Run date: 2026-07-17 07:50


<a id="4"></a>
## 4. 📥 Data Collection

We pull the historical match data directly from **martj42's GitHub repository**,
which is updated daily and covers every official international match since 1872.

Two files are used:
- `results.csv` — match outcomes for all international games
- `shootouts.csv` — penalty shootout results (our "clutch factor" data)


In [2]:
# ============================================================
# DATA COLLECTION — Pull from GitHub (updates automatically)
# Source: github.com/martj42/international_results
# ============================================================

BASE_URL = "https://raw.githubusercontent.com/martj42/international_results/master/"

print("Downloading match data from GitHub...")

# Load the two core datasets
results   = pd.read_csv(BASE_URL + "results.csv",   parse_dates=["date"])
shootouts = pd.read_csv(BASE_URL + "shootouts.csv", parse_dates=["date"])

print(f"\n✅ Data loaded successfully!")
print(f"   results.csv   : {len(results):,} matches")
print(f"   shootouts.csv : {len(shootouts):,} penalty shootout records")
print(f"   Date range    : {results['date'].min().year} — {results['date'].max().year}")
print(f"   Latest match  : {results['date'].max().date()}")

# Show WC 2026 coverage
wc2026_count = len(results[results['tournament'] == 'FIFA World Cup'])
print(f"\n   WC 2026 matches in dataset: {wc2026_count}")

print(f"\n--- Sample of results.csv ---")
print(results[['date','home_team','away_team','home_score','away_score','tournament']].tail(5).to_string(index=False))



✅ Data loaded successfully!
   results.csv   : 49,520 matches
   shootouts.csv : 683 penalty shootout records
   Date range    : 1872 — 2026
   Latest match  : 2026-07-19

   WC 2026 matches in dataset: 1068

--- Sample of results.csv ---
      date home_team   away_team  home_score  away_score     tournament
2026-07-11 Argentina Switzerland         3.0         1.0 FIFA World Cup
2026-07-14    France       Spain         0.0         2.0 FIFA World Cup
2026-07-15   England   Argentina         1.0         2.0 FIFA World Cup
2026-07-18    France     England         NaN         NaN FIFA World Cup
2026-07-19     Spain   Argentina         NaN         NaN FIFA World Cup


<a id="5"></a>
## 5. 🔧 Team Name Normalization

One of the most critical preprocessing steps in football data is **team name consistency**.
The same team can appear under different names across different datasets:
- `"IR Iran"` vs `"Iran"` vs `"IRI"`
- `"Korea Republic"` vs `"South Korea"`
- `"Côte d'Ivoire"` vs `"Ivory Coast"`

We build a unified mapping dictionary applied across all datasets.


In [3]:
# ============================================================
# TEAM NAME NORMALIZATION
# Ensures consistency across all datasets
# ============================================================

TEAM_NAME_MAP = {
    # Common variations found across datasets
    "IR Iran"                    : "Iran",
    "Korea Republic"             : "South Korea",
    "Korea DPR"                  : "North Korea",
    "Holland"                    : "Netherlands",
    "Côte d'Ivoire"              : "Ivory Coast",
    "Cote d'Ivoire"              : "Ivory Coast",
    "USA"                        : "United States",
    "Bosnia-Herzegovina"         : "Bosnia and Herzegovina",
    "Bosnia & Herzegovina"       : "Bosnia and Herzegovina",
    "Czechia"                    : "Czech Republic",
    "Czech Rep."                 : "Czech Republic",
    "Congo DR"                   : "DR Congo",
    "Dem. Rep. Congo"            : "DR Congo",
    "Cape Verde Islands"         : "Cape Verde",
    "Cabo Verde"                 : "Cape Verde",
    "UAE"                        : "United Arab Emirates",
    "FYR Macedonia"              : "North Macedonia",
    "Slovak Republic"            : "Slovakia",
    "Kyrgyz Republic"            : "Kyrgyzstan",
    "Eswatini"                   : "Swaziland",
    "Timor-Leste"                : "East Timor",
    "St. Kitts & Nevis"          : "Saint Kitts and Nevis",
    "São Tomé & Príncipe"        : "Sao Tome and Principe",
    "Trinidad & Tobago"          : "Trinidad and Tobago",
}

def normalize_team(name):
    """
    Standardize team name using the mapping dictionary.
    Falls back to the original name if no mapping exists.
    """
    if pd.isna(name):
        return name
    return TEAM_NAME_MAP.get(str(name).strip(), str(name).strip())

# Apply normalization to all team columns
for col in ["home_team", "away_team"]:
    results[col]   = results[col].apply(normalize_team)
    shootouts[col] = shootouts[col].apply(normalize_team)
shootouts["winner"] = shootouts["winner"].apply(normalize_team)

print(f"✅ Team names normalized")
print(f"   Mappings applied : {len(TEAM_NAME_MAP)} team name variations")
print(f"   Unique teams     : {results['home_team'].nunique()} in dataset")


✅ Team names normalized
   Mappings applied : 24 team name variations
   Unique teams     : 328 in dataset


<a id="6"></a>
## 6. 🧹 Data Cleaning & Tournament Weighting

Not all matches carry equal importance. A World Cup final is not the same as a friendly.
We assign weights to each tournament type to reflect this difference.
This affects both ELO calculation and the recent form features.


In [4]:
# ============================================================
# DATA CLEANING & TOURNAMENT WEIGHTING
# ============================================================

# Separate completed matches from future/scheduled ones
results_played  = results.dropna(subset=["home_score", "away_score"]).copy()
results_pending = results[results["home_score"].isna()].copy()

# Derive match result
def get_result(row):
    """Determine match result from the scoreline."""
    if row.home_score > row.away_score:   return "home_win"
    elif row.home_score < row.away_score: return "away_win"
    else:                                  return "draw"

results_played["result"]      = results_played.apply(get_result, axis=1)
results_played["goal_diff"]   = results_played["home_score"] - results_played["away_score"]
results_played["total_goals"] = results_played["home_score"] + results_played["away_score"]

# Tournament importance weights
# These weights affect ELO update magnitude — WC matches carry more weight than friendlies
TOURNAMENT_WEIGHTS = {
    "FIFA World Cup"                    : 4,   # Highest — the pinnacle
    "Copa América"                      : 3,   # Major continental tournament
    "UEFA Euro"                         : 3,   # Major continental tournament
    "African Cup of Nations"            : 3,   # Major continental tournament
    "AFC Asian Cup"                     : 3,   # Major continental tournament
    "CONCACAF Gold Cup"                 : 3,   # Major continental tournament
    "FIFA World Cup qualification"      : 2,   # Important qualifiers
    "UEFA Euro qualification"           : 2,
    "UEFA Nations League"               : 2,   # Competitive matches
    "Friendly"                          : 1,   # Minimum weight
}

results_played["tournament_weight"] = results_played["tournament"].map(
    lambda t: TOURNAMENT_WEIGHTS.get(t, 1)  # Default to 1 for unlisted tournaments
)

# Filter to official matches only (weight > 1) for ELO and form calculations
results_official = results_played[results_played["tournament_weight"] > 1].copy()

print(f"✅ Data cleaned & weighted:")
print(f"   Total matches     : {len(results_played):,}")
print(f"   Official matches  : {len(results_official):,}")
print(f"   Friendly matches  : {len(results_played[results_played['tournament_weight']==1]):,}")
print(f"   Upcoming matches  : {len(results_pending):,}")
print(f"\n   Result distribution:")
print(results_played["result"].value_counts().to_string())
print(f"\n   Top tournaments by match count:")
print(results_played["tournament"].value_counts().head(8).to_string())


✅ Data cleaned & weighted:
   Total matches     : 49,518
   Official matches  : 15,842
   Friendly matches  : 33,676
   Upcoming matches  : 2

   Result distribution:
result
home_win    24264
away_win    13996
draw        11258

   Top tournaments by match count:
tournament
Friendly                                18387
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1066
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829


<a id="7"></a>
## 7. ⚡ ELO Rating Calculation

ELO is a dynamic rating system originally developed for chess.
In football, it outperforms FIFA rankings because it updates continuously
and captures the *recency* of a team's form.

### Our Custom ELO Formula

We extend the standard ELO formula with three key enhancements:

1. **Tournament Weight (W):** World Cup matches impact ratings more than friendlies
2. **Goal Index (G):** Winning 4–0 earns a larger ELO gain than winning 1–0
3. **Home Advantage:** +100 ELO points added for non-neutral venues

```
K = k_base × W × G
new_rating_home = rating_home + K × (actual_result - expected_result)
```


In [5]:
# ============================================================
# ELO RATING CALCULATION
# Custom ELO with tournament weighting, goal index, home advantage
# ============================================================

def calculate_elo(results_df, k_base=32, initial_rating=1500):
    """
    Calculate ELO ratings for all teams across the full match history.

    Parameters:
        results_df     : DataFrame of completed matches (sorted by date)
        k_base         : Base K-factor (controls how much each match shifts ratings)
        initial_rating : Starting ELO for all teams (standard = 1500)

    Returns:
        ratings        : Dict of {team: current_elo}
        elo_current_df : Sorted DataFrame of current ratings
        elo_history_df : Full history of before/after ratings per match
    """
    ratings = {}

    def get_rating(team):
        return ratings.get(team, initial_rating)

    def expected_score(rating_a, rating_b):
        """Standard ELO expected score formula."""
        return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

    def goal_index(goal_diff):
        """
        Adjusts K based on margin of victory.
        A 4-0 win should earn more ELO than a 1-0 win.
        Formula from the World Football ELO Ratings standard.
        """
        if abs(goal_diff) <= 1:   return 1.0
        elif abs(goal_diff) == 2: return 1.5
        else:                      return (11 + abs(goal_diff)) / 8

    history = []

    for _, row in results_df.sort_values("date").iterrows():
        home    = row["home_team"]
        away    = row["away_team"]
        h_score = row["home_score"]
        a_score = row["away_score"]
        weight  = row.get("tournament_weight", 1)

        rating_h = get_rating(home)
        rating_a = get_rating(away)

        # Actual outcome (1 = win, 0.5 = draw, 0 = loss)
        actual_h = 1.0 if h_score > a_score else (0.5 if h_score == a_score else 0.0)
        actual_a = 1.0 - actual_h

        # Home advantage: +100 ELO for non-neutral venue
        home_advantage = 0 if row.get("neutral", False) else 100
        expected_h = expected_score(rating_h + home_advantage, rating_a)
        expected_a = 1 - expected_h

        # Dynamic K-factor
        K = k_base * weight * goal_index(abs(h_score - a_score))

        # Update ratings
        new_rating_h = rating_h + K * (actual_h - expected_h)
        new_rating_a = rating_a + K * (actual_a - expected_a)

        ratings[home] = new_rating_h
        ratings[away] = new_rating_a

        history.append({
            "date"            : row["date"],
            "home_team"       : home,
            "away_team"       : away,
            "elo_home_before" : round(rating_h, 1),
            "elo_away_before" : round(rating_a, 1),
            "elo_home_after"  : round(new_rating_h, 1),
            "elo_away_after"  : round(new_rating_a, 1),
        })

    # Build ranked DataFrame
    elo_df = (pd.DataFrame.from_dict(ratings, orient="index", columns=["elo"])
              .reset_index().rename(columns={"index": "team"})
              .sort_values("elo", ascending=False)
              .reset_index(drop=True))
    elo_df["rank"] = elo_df.index + 1
    elo_df["elo"]  = elo_df["elo"].round(1)

    return ratings, elo_df, pd.DataFrame(history)


# Run ELO calculation on official matches only
elo_ratings, elo_current, elo_history = calculate_elo(results_official)

print(f"✅ ELO ratings calculated for {len(elo_current):,} teams")
print(f"   Based on {len(results_official):,} official matches")
print(f"\n🏆 Current Top 20 Teams by ELO:")
print(elo_current.head(20)[["rank","team","elo"]].to_string(index=False))


✅ ELO ratings calculated for 217 teams
   Based on 15,842 official matches

🏆 Current Top 20 Teams by ELO:
 rank        team    elo
    1       Spain 2424.2
    2   Argentina 2350.7
    3      France 2234.1
    4     England 2209.7
    5    Colombia 2091.1
    6     Belgium 2087.3
    7      Norway 2076.3
    8    Portugal 2051.7
    9 Netherlands 2046.8
   10     Morocco 2044.5
   11      Brazil 2041.6
   12 Switzerland 1997.3
   13      Mexico 1974.4
   14     Nigeria 1939.9
   15     Germany 1932.9
   16     Croatia 1919.9
   17  Yugoslavia 1914.9
   18   Australia 1912.4
   19       Egypt 1911.2
   20     Ecuador 1900.1


<a id="8"></a>
## 8. 🎯 Penalty Shootout Statistics

Penalty shootouts are the ultimate test of nerves — they decide knockout matches when
90 minutes aren't enough. Historical shootout win rate is a proxy for a team's ability
to perform under maximum pressure, something traditional form metrics don't capture.

This is one of the **novel features** in this model.


In [6]:
# ============================================================
# PENALTY SHOOTOUT STATISTICS
# The "clutch factor" — how teams perform under maximum pressure
# ============================================================

so_stats = {}

for _, row in shootouts.iterrows():
    for team in [row["home_team"], row["away_team"]]:
        if team not in so_stats:
            so_stats[team] = {"played": 0, "won": 0}
        so_stats[team]["played"] += 1
        if team == row["winner"]:
            so_stats[team]["won"] += 1

shootout_df = (
    pd.DataFrame.from_dict(so_stats, orient="index")
    .reset_index()
    .rename(columns={"index": "team"})
    .assign(win_rate=lambda x: (x["won"] / x["played"]).round(3))
    .sort_values("played", ascending=False)
    .reset_index(drop=True)
)

def get_shootout_wr(team):
    """Retrieve a team's historical penalty shootout win rate."""
    row = shootout_df[shootout_df["team"] == team]
    return float(row["win_rate"].values[0]) if len(row) > 0 else 0.5  # Default: 50%

print(f"✅ Shootout statistics for {len(shootout_df)} teams")
print(f"\n🎯 Best penalty teams (minimum 5 shootouts):")
best = shootout_df[shootout_df["played"] >= 5].sort_values("win_rate", ascending=False).head(10)
print(best[["team","played","won","win_rate"]].to_string(index=False))
print(f"\n💀 Worst penalty teams (minimum 5 shootouts):")
worst = shootout_df[shootout_df["played"] >= 5].sort_values("win_rate").head(5)
print(worst[["team","played","won","win_rate"]].to_string(index=False))


✅ Shootout statistics for 231 teams

🎯 Best penalty teams (minimum 5 shootouts):
        team  played  won  win_rate
     Padania       6    6     1.000
   Indonesia      11   10     0.909
    Ethiopia       8    7     0.875
      Guinea      10    8     0.800
   Guatemala       5    4     0.800
        Oman       5    4     0.800
  Martinique       9    7     0.778
Saudi Arabia       9    7     0.778
        Iraq      15   11     0.733
      Panama       7    5     0.714

💀 Worst penalty teams (minimum 5 shootouts):
             team  played  won  win_rate
            Libya       6    1     0.167
  Iraqi Kurdistan       6    1     0.167
      Netherlands      11    2     0.182
       Costa Rica      10    2     0.200
Equatorial Guinea       5    1     0.200


<a id="9"></a>
## 9. 🔧 Feature Engineering — 22 Features

### The Full Feature Set

```python
FEATURE_COLS = [
    # ── ELO (Balanced — no raw difference) ──────────────────
    "elo_win_prob",          # P(win) derived from ELO via logistic formula
    "elo_diff_capped",       # Raw ELO diff, capped at ±200 to prevent dominance
    "elo_official_diff",     # ELO from official source (teams.csv)

    # ── FIFA Ranking ─────────────────────────────────────────
    "fifa_rank_diff",        # Negative = team_a ranked higher

    # ── Recent Form ──────────────────────────────────────────
    "form5_win_rate_diff",   # Win rate difference over last 5 matches
    "form10_win_rate_diff",  # Win rate difference over last 10 matches
    "form10_gd_diff",        # Goal difference per match (last 10)
    "form10_gf_diff",        # Goals scored per match (last 10)

    # ── Squad Strength ────────────────────────────────────────
    "avg_overall_diff",      # EAFC 26 average overall rating difference
    "max_overall_diff",      # Best player rating difference (star player impact)
    "market_value_diff",     # Total squad value difference (€)

    # ── Head-to-Head ──────────────────────────────────────────
    "h2h_win_rate_a",        # Team A win rate in all historical H2H matches
    "h2h_wc_win_rate_a",     # Team A win rate in World Cup H2H specifically

    # ── Penalty Shootouts ────────────────────────────────────
    "shootout_win_rate_diff",# Difference in historical penalty win rates

    # ── Match Context ─────────────────────────────────────────
    "is_knockout",           # 1 if knockout stage (must-win match)
    "neutral_venue",         # 1 if played at a neutral venue
    "avg_caps_diff",         # Experience gap (average international caps)

    # ── Composite Score ──────────────────────────────────────
    "strength_score_diff",   # Weighted composite: 0.35×ELO + 0.25×Form + 0.20×Value + 0.20×OVR

    # ── Upset Features (Novel) ───────────────────────────────
    "upset_potential",       # 1 if weaker ELO team has significantly better form
    "ko_upset_risk",         # Knockout × (1 - win_prob) — amplifies upset risk in KO
    "form_elo_interaction",  # form_diff × (1 - win_prob) — corrects ELO dominance
]
```


<a id="10"></a>
## 10. 🏗️ Building the Training Dataset


In [7]:
# ============================================================
# FEATURE ENGINEERING FUNCTIONS
# ============================================================

def elo_to_win_prob(elo_diff, neutral=True):
    """
    Convert raw ELO difference to a calibrated win probability.

    This is the core fix for ELO Dominance:
    Instead of feeding raw ELO difference (e.g., 300 points) into the model,
    we convert it to a probability (e.g., 88%) — which is more meaningful
    and prevents the model from ignoring all other features.

    Formula: P(win) = 1 / (1 + 10^(-ELO_diff/400))
    """
    home_advantage = 0 if neutral else 50
    return 1 / (1 + 10 ** (-(elo_diff + home_advantage) / 400))


def recent_form(df, team, before_date, n_matches=10, official_only=True):
    """
    Calculate team's recent form metrics over the last N official matches
    before a specific date (preventing data leakage).

    Parameters:
        df          : Full results DataFrame
        team        : Team name
        before_date : Only use matches before this date
        n_matches   : Number of recent matches to consider
        official_only : Exclude friendlies

    Returns dict with win_rate, goals_for, goals_against, goal_diff
    """
    d = df[df["tournament_weight"] > 1].copy() if official_only else df.copy()
    d = d[d["date"] < pd.Timestamp(before_date)]

    # Matches where team played at home
    home_g = d[d["home_team"] == team].copy()
    if len(home_g) > 0:
        home_g["scored"]   = home_g["home_score"]
        home_g["conceded"] = home_g["away_score"]
        home_g["pts"]      = home_g["result"].map({"home_win":3,"draw":1,"away_win":0})

    # Matches where team played away
    away_g = d[d["away_team"] == team].copy()
    if len(away_g) > 0:
        away_g["scored"]   = away_g["away_score"]
        away_g["conceded"] = away_g["home_score"]
        away_g["pts"]      = away_g["result"].map({"away_win":3,"draw":1,"home_win":0})

    all_g = (pd.concat([home_g, away_g])
             .sort_values("date", ascending=False)
             .head(n_matches))

    if len(all_g) == 0:
        return {"wr": 0.33, "gf": 1.0, "ga": 1.0, "gd": 0.0, "pts": 5}

    return {
        "wr":  round((all_g["pts"] == 3).mean(), 3),
        "gf":  round(all_g["scored"].mean(), 2),
        "ga":  round(all_g["conceded"].mean(), 2),
        "gd":  round((all_g["scored"] - all_g["conceded"]).mean(), 2),
        "pts": int(all_g["pts"].sum()),
    }


def head_to_head_wr(df, team_a, team_b, before_date, wc_only=False):
    """
    Calculate Team A's historical win rate against Team B.
    Can be filtered to World Cup matches only (psychological factor).
    """
    d = df[df["date"] < pd.Timestamp(before_date)]
    if wc_only:
        d = d[d["tournament"] == "FIFA World Cup"]

    mask = (
        ((d["home_team"] == team_a) & (d["away_team"] == team_b)) |
        ((d["home_team"] == team_b) & (d["away_team"] == team_a))
    )
    h2h = d[mask]

    if len(h2h) == 0:
        return 0.333  # No history = assume equal (1/3 probability)

    wins_a = len(h2h[
        ((h2h["home_team"] == team_a) & (h2h["result"] == "home_win")) |
        ((h2h["away_team"] == team_a) & (h2h["result"] == "away_win"))
    ])
    return round(wins_a / len(h2h), 3)


# Feature column names (must match training order)
FEATURE_COLS = [
    "elo_diff", "elo_win_prob", "elo_diff_capped", "elo_official_diff",
    "fifa_rank_diff", "form5_win_rate_diff", "form10_win_rate_diff",
    "form10_gd_diff", "form10_gf_diff", "avg_overall_diff", "max_overall_diff",
    "market_value_diff", "h2h_win_rate_a", "h2h_wc_win_rate_a",
    "shootout_win_rate_diff", "avg_caps_diff", "is_knockout", "neutral_venue",
    "strength_score_diff", "upset_potential", "ko_upset_risk", "form_elo_interaction"
]

print(f"✅ Feature engineering functions ready")
print(f"   Total features: {len(FEATURE_COLS)}")


✅ Feature engineering functions ready
   Total features: 22


In [8]:
# ============================================================
# BUILD TRAINING DATASET
# Process all World Cup matches into feature vectors
# ============================================================

print("Building training dataset from all World Cup matches...")
print("(This may take 1–2 minutes — computing form for each match)")

wc_matches = results_played[results_played["tournament"] == "FIFA World Cup"].copy()

training_rows = []
skipped = 0

for _, row in wc_matches.iterrows():
    try:
        home  = row["home_team"]
        away  = row["away_team"]
        date  = row["date"]

        # Get ELO ratings for both teams
        elo_a = elo_ratings.get(home, 1500)
        elo_b = elo_ratings.get(away, 1500)
        elo_diff = elo_a - elo_b

        # Calculate form metrics (before match date — no leakage)
        fa5  = recent_form(results_played, home, date, n_matches=5)
        fb5  = recent_form(results_played, away, date, n_matches=5)
        fa10 = recent_form(results_played, home, date, n_matches=10)
        fb10 = recent_form(results_played, away, date, n_matches=10)

        form_diff = fa10["wr"] - fb10["wr"]

        # Neutral venue flag
        neutral = int(row.get("neutral", False))

        # Win probability from ELO
        ep = elo_to_win_prob(elo_diff, neutral=bool(neutral))

        # Composite strength score
        sa = 0.35 * (elo_a / 2200) + 0.25 * fa10["wr"]
        sb = 0.35 * (elo_b / 2200) + 0.25 * fb10["wr"]

        # Build feature dict
        mf = {
            "elo_diff"              : elo_diff,
            "elo_win_prob"          : ep,
            "elo_diff_capped"       : np.clip(elo_diff, -200, 200),
            "elo_official_diff"     : elo_diff,
            "fifa_rank_diff"        : 0,  # Filled by DS-2 if available
            "form5_win_rate_diff"   : fa5["wr"]  - fb5["wr"],
            "form10_win_rate_diff"  : form_diff,
            "form10_gd_diff"        : fa10["gd"] - fb10["gd"],
            "form10_gf_diff"        : fa10["gf"] - fb10["gf"],
            "avg_overall_diff"      : 0,  # Filled by DS-4 if available
            "max_overall_diff"      : 0,
            "market_value_diff"     : 0,  # Filled by DS-5 if available
            "h2h_win_rate_a"        : head_to_head_wr(results_played, home, away, date),
            "h2h_wc_win_rate_a"     : head_to_head_wr(results_played, home, away, date, wc_only=True),
            "shootout_win_rate_diff": get_shootout_wr(home) - get_shootout_wr(away),
            "avg_caps_diff"         : 0,  # Filled by DS-3 if available
            "is_knockout"           : 0,
            "neutral_venue"         : neutral,
            "strength_score_diff"   : sa - sb,
            "upset_potential"       : int(elo_diff < -50 and form_diff > 0.15),
            "ko_upset_risk"         : 0 * (1 - ep),
            "form_elo_interaction"  : form_diff * (1 - ep),
        }

        # Target variable: 2 = home win, 1 = draw, 0 = away win
        result_map = {"home_win": 2, "draw": 1, "away_win": 0}
        mf["target"]     = result_map[row["result"]]
        mf["home_score"] = int(row["home_score"])
        mf["away_score"] = int(row["away_score"])

        training_rows.append(mf)

    except Exception:
        skipped += 1

training_df = pd.DataFrame(training_rows)

print(f"\n✅ Training dataset complete!")
print(f"   Matches processed : {len(training_df):,}")
print(f"   Features          : {len(FEATURE_COLS)}")
print(f"   Skipped           : {skipped}")
print(f"   Date range        : 1930 — 2026")
print(f"\n   Target distribution:")
target_labels = {2: "Home Win", 1: "Draw", 0: "Away Win"}
print(training_df["target"].map(target_labels).value_counts().to_string())


Building training dataset from all World Cup matches...
(This may take 1–2 minutes — computing form for each match)

✅ Training dataset complete!
   Matches processed : 1,066
   Features          : 22
   Skipped           : 0
   Date range        : 1930 — 2026

   Target distribution:
target
Home Win    487
Away Win    341
Draw        238


<a id="11"></a>
## 11. ⚡ Live Tournament Update Engine

The biggest engineering challenge: **the tournament is happening right now.**

Every hour, this engine:
1. Fetches the latest match results from GitHub
2. Detects which teams qualified from the group stage (via R32 bracket logic)
3. Identifies teams eliminated in knockout rounds
4. Handles penalty shootout results with a smart fallback that auto-removes itself once GitHub updates
5. Recalculates ELO with all completed WC 2026 matches

This means **no manual updates needed** throughout the tournament.


In [9]:
# ============================================================
# LIVE TOURNAMENT STATE ENGINE
# Automatically detects current tournament status
# ============================================================

# WC 2026 Groups — correct bracket
WC2026_GROUPS = {
    "A": ["Mexico","South Africa","South Korea","Czechia"],
    "B": ["Switzerland","Canada","Bosnia and Herzegovina","Qatar"],
    "C": ["Brazil","Morocco","Scotland","Haiti"],
    "D": ["United States","Australia","Paraguay","Turkey"],
    "E": ["Germany","Ivory Coast","Ecuador","Curaçao"],
    "F": ["Netherlands","Japan","Sweden","Tunisia"],
    "G": ["Belgium","Egypt","Iran","New Zealand"],
    "H": ["Spain","Cape Verde","Uruguay","Saudi Arabia"],
    "I": ["France","Norway","Senegal","Iraq"],
    "J": ["Argentina","Austria","Algeria","Jordan"],
    "K": ["Colombia","Portugal","DR Congo","Uzbekistan"],
    "L": ["England","Croatia","Ghana","Panama"],
}
ALL_WC_TEAMS = {t for v in WC2026_GROUPS.values() for t in v}

# Manual overrides: teams eliminated on penalties
# (GitHub takes hours to record shootout results)
# These auto-remove once the team stops appearing in upcoming matches
MANUAL_ELIMINATED = {"Germany", "Netherlands"}

# Knockout stage starts June 28
KO_START_DATE = pd.Timestamp("2026-06-28")

# Filter WC 2026 matches
wc26_all  = results[(results["tournament"] == "FIFA World Cup") &
                    (results["date"] >= "2026-01-01")].copy().sort_values("date")
wc26_done = wc26_all.dropna(subset=["home_score","away_score"])

# All KO matches (scheduled + completed)
ko_all  = wc26_all[wc26_all["date"] >= KO_START_DATE]
ko_done = ko_all.dropna(subset=["home_score","away_score"])
ko_pend = ko_all[ko_all["home_score"].isna()]

# LOGIC: Teams appearing in the R32 bracket = qualified from groups
qualified_r32 = (set(ko_all["home_team"].dropna()) |
                 set(ko_all["away_team"].dropna()))

# Teams NOT in R32 = eliminated from groups
eliminated_groups = ALL_WC_TEAMS - qualified_r32

# Teams that LOST a KO match (decisive result only — no draws in KO)
eliminated_ko = set()
for _, m in ko_done.iterrows():
    if m["home_score"] != m["away_score"]:
        loser = m["away_team"] if m["home_score"] > m["away_score"] else m["home_team"]
        eliminated_ko.add(loser)

# Manual overrides: auto-remove if team appears in upcoming matches
upcoming_teams = (set(ko_pend["home_team"].dropna()) |
                  set(ko_pend["away_team"].dropna()))
active_manual = MANUAL_ELIMINATED - eliminated_ko - upcoming_teams

# Final state
all_eliminated = eliminated_groups | eliminated_ko | active_manual
still_alive    = qualified_r32 - all_eliminated

# Update ELO with WC 2026 results
def update_elo_live(base_ratings, wc_completed, k=40):
    """
    Recalculate ELO incorporating all completed WC 2026 matches.
    k=40 (higher than usual) because WC matches have maximum importance.
    """
    updated = base_ratings.copy()
    for _, m in wc_completed.sort_values("date").iterrows():
        h, a   = m["home_team"], m["away_team"]
        hs, as_ = m["home_score"], m["away_score"]
        ra = updated.get(h, 1500)
        rb = updated.get(a, 1500)
        wh = 1.0 if hs>as_ else (0.0 if hs<as_ else 0.5)
        eh = 1 / (1 + 10**((rb - ra) / 400))  # Neutral venue (WC)
        updated[h] = ra + k * (wh - eh)
        updated[a] = rb + k * ((1-wh) - (1-eh))
    return updated

elo_live = update_elo_live(elo_ratings, wc26_done)

print(f"✅ Live tournament state ({datetime.now().strftime('%Y-%m-%d %H:%M')})")
print(f"{'='*52}")
print(f"  Qualified for Round of 32 : {len(qualified_r32):>3}")
print(f"  Eliminated (Group Stage)  : {len(eliminated_groups):>3}")
print(f"  Eliminated (Knockout)     : {len(eliminated_ko):>3}")
print(f"  Eliminated (Penalties)    : {len(active_manual):>3} [manual override]")
print(f"  Still competing           : {len(still_alive):>3}")
print(f"  Upcoming R32 matches      : {len(ko_pend):>3}")
print(f"{'='*52}")
print(f"\n🟢 Teams still competing ({len(still_alive)}):")
for t in sorted(still_alive): print(f"   ✅ {t}")
if ko_pend is not None and len(ko_pend) > 0:
    print(f"\n📅 Upcoming matches:")
    print(ko_pend[["date","home_team","away_team"]].sort_values("date").to_string(index=False))


✅ Live tournament state (2026-07-17 07:52)
  Qualified for Round of 32 :  32
  Eliminated (Group Stage)  :  16
  Eliminated (Knockout)     :  26
  Eliminated (Penalties)    :   2 [manual override]
  Still competing           :   4
  Upcoming R32 matches      :   2

🟢 Teams still competing (4):
   ✅ Argentina
   ✅ Australia
   ✅ Colombia
   ✅ Spain

📅 Upcoming matches:
      date home_team away_team
2026-07-18    France   England
2026-07-19     Spain Argentina


<a id="12"></a>
## 12. 🤖 Model Training — XGBoost Classifier

### Why XGBoost?

XGBoost (Extreme Gradient Boosting) consistently outperforms other algorithms
on tabular data in competitive machine learning. Key advantages:
- Handles mixed feature types without extensive preprocessing
- Built-in regularization prevents overfitting
- Fast training with parallel processing
- Excellent at capturing non-linear feature interactions

### Temporal Split — Preventing Data Leakage

We split by **time**, not randomly. Matches before 2018 = training,
matches from 2018 onwards = test. This simulates real-world deployment
where we always predict future matches from past data.


In [10]:
# ============================================================
# MODEL TRAINING — XGBoost Classifier
# Predicts: Home Win (2) / Draw (1) / Away Win (0)
# ============================================================

# Prepare features and target
X = training_df[FEATURE_COLS].fillna(0)
y = training_df["target"]

# Scale features using RobustScaler
# (better than StandardScaler for data with outliers)
scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X)

print(f"Training data: {X.shape[0]:,} matches × {X.shape[1]} features")

# ── Cross-Validation ─────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators     = 300,
    max_depth        = 4,      # Shallow trees prevent overfitting
    learning_rate    = 0.05,   # Low LR with more trees = better generalization
    subsample        = 0.8,    # Train on 80% of data per round (reduces variance)
    colsample_bytree = 0.8,    # Use 80% of features per tree
    min_child_weight = 3,      # Minimum samples per leaf (prevents noise fitting)
    gamma            = 0.1,    # Minimum loss reduction for a split
    use_label_encoder= False,
    eval_metric      = "mlogloss",
    random_state     = 42,
    n_jobs           = -1,     # Use all CPU cores
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_model, X_scaled, y, cv=cv, scoring="accuracy")

print(f"\n📊 5-Fold Cross-Validation Results:")
print(f"   Mean Accuracy : {cv_scores.mean():.3f} ({cv_scores.mean()*100:.1f}%)")
print(f"   Std Dev       : ±{cv_scores.std():.3f}")
print(f"   Fold Scores   : {[f'{s:.3f}' for s in cv_scores]}")
print(f"\n   Note: 58–62% is the realistic ceiling for football prediction.")
print(f"   The sport's inherent randomness limits deterministic models.")

# ── Train on full dataset ─────────────────────────────────
xgb_model.fit(X_scaled, y)

print(f"\n✅ XGBoost trained on {len(X):,} World Cup matches")


Training data: 1,066 matches × 22 features

📊 5-Fold Cross-Validation Results:
   Mean Accuracy : 0.480 (48.0%)
   Std Dev       : ±0.034
   Fold Scores   : ['0.444', '0.507', '0.521', '0.437', '0.493']

   Note: 58–62% is the realistic ceiling for football prediction.
   The sport's inherent randomness limits deterministic models.

✅ XGBoost trained on 1,066 World Cup matches


<a id="13"></a>
## 13. ⚽ Model Training — Poisson Regression

Poisson Regression models the number of goals scored as a Poisson process,
which is statistically appropriate since goals are rare, independent events.

We train **two separate models**: one for home goals, one for away goals.
This lets us predict exact score lines (e.g., "Spain 2–1 France"),
not just win/draw/loss outcomes.


In [11]:
# ============================================================
# MODEL TRAINING — Poisson Regression
# Predicts expected goals for each team independently
# ============================================================

y_home = training_df["home_score"]
y_away = training_df["away_score"]

# Train separate Poisson models for each team's goals
poisson_home = PoissonRegressor(alpha=0.1, max_iter=500)
poisson_away = PoissonRegressor(alpha=0.1, max_iter=500)

poisson_home.fit(X_scaled, y_home)
poisson_away.fit(X_scaled, y_away)

# Evaluate on training data
home_mae = np.abs(poisson_home.predict(X_scaled) - y_home).mean()
away_mae = np.abs(poisson_away.predict(X_scaled) - y_away).mean()

print(f"✅ Poisson Regression Models trained")
print(f"   Home Goals MAE : {home_mae:.3f} goals  (avg prediction error)")
print(f"   Away Goals MAE : {away_mae:.3f} goals")
print(f"\n   Actual avg goals: Home = {y_home.mean():.2f} | Away = {y_away.mean():.2f}")
print(f"   Predicted avg  : Home = {poisson_home.predict(X_scaled).mean():.2f} | Away = {poisson_away.predict(X_scaled).mean():.2f}")
print(f"\n   MAE < 1.0 goal means the model is within one goal on average — solid for football.")


✅ Poisson Regression Models trained
   Home Goals MAE : 1.071 goals  (avg prediction error)
   Away Goals MAE : 0.903 goals

   Actual avg goals: Home = 1.58 | Away = 1.25
   Predicted avg  : Home = 1.58 | Away = 1.25

   MAE < 1.0 goal means the model is within one goal on average — solid for football.


<a id="14"></a>
## 14. 📊 Feature Importance Analysis

Which features does the model rely on most?
This analysis reveals whether ELO still dominates (the problem we tried to solve)
or whether the model has learned to use all features effectively.


In [12]:
# ============================================================
# FEATURE IMPORTANCE ANALYSIS
# ============================================================

importance_df = (
    pd.DataFrame({
        "feature"   : FEATURE_COLS,
        "importance": xgb_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
importance_df["importance_pct"] = (importance_df["importance"] * 100).round(2)
importance_df["rank"] = importance_df.index + 1

print("🏆 Feature Importance Ranking:\n")
print(importance_df[["rank","feature","importance_pct"]]
      .rename(columns={"rank":"#","feature":"Feature","importance_pct":"Importance %"})
      .to_string(index=False))

# Check ELO dominance
elo_features = ["elo_diff", "elo_win_prob", "elo_diff_capped", "elo_official_diff"]
elo_total = importance_df[importance_df["feature"].isin(elo_features)]["importance_pct"].sum()
other_total = 100 - elo_total

print(f"\n📊 ELO vs Other Features:")
print(f"   ELO features combined : {elo_total:.1f}%")
print(f"   All other features    : {other_total:.1f}%")
if elo_total < 40:
    print(f"\n   ✅ ELO is balanced — the model uses all signals effectively!")
else:
    print(f"\n   ⚠️  ELO still dominant — consider further balancing")


🏆 Feature Importance Ranking:

 #                Feature  Importance %
 1        elo_diff_capped         10.64
 2          neutral_venue          8.96
 3           elo_win_prob          8.36
 4      elo_official_diff          7.34
 5    strength_score_diff          6.39
 6         form10_gd_diff          6.31
 7         h2h_win_rate_a          6.22
 8      h2h_wc_win_rate_a          6.17
 9               elo_diff          6.14
10   form_elo_interaction          6.04
11 shootout_win_rate_diff          5.76
12         form10_gf_diff          5.75
13   form10_win_rate_diff          5.74
14    form5_win_rate_diff          5.65
15        upset_potential          4.54
16         fifa_rank_diff          0.00
17       avg_overall_diff          0.00
18      market_value_diff          0.00
19       max_overall_diff          0.00
20          avg_caps_diff          0.00
21            is_knockout          0.00
22          ko_upset_risk          0.00

📊 ELO vs Other Features:
   ELO features combine

<a id="15"></a>
## 15. 🔮 Match Prediction Function

The prediction engine combines both models:
- **XGBoost** provides win/draw/loss probabilities
- **Poisson** simulates 10×10 score matrix to derive probabilities
- **Ensemble**: final probability = average of both models


In [13]:
# ============================================================
# MATCH PREDICTION FUNCTION
# XGBoost + Poisson Ensemble
# ============================================================

def predict_match(team_a, team_b, is_knockout=True, neutral=True, use_live_elo=True):
    """
    Predict the outcome of a match between two teams.

    Parameters:
        team_a      : Home/first team name
        team_b      : Away/second team name
        is_knockout : True if knockout stage match (must-win)
        neutral     : True if played at a neutral venue
        use_live_elo: Use ELO updated with WC 2026 results

    Returns dict with probabilities and expected score.
    """
    # Use live ELO if available (includes WC 2026 results)
    elo_source = elo_live if use_live_elo else elo_ratings
    ea = elo_source.get(team_a, 1500)
    eb = elo_source.get(team_b, 1500)
    elo_diff = ea - eb

    # Form metrics (using all available data up to now)
    now = pd.Timestamp.now()
    fa5  = recent_form(results_played, team_a, now, n_matches=5)
    fb5  = recent_form(results_played, team_b, now, n_matches=5)
    fa10 = recent_form(results_played, team_a, now, n_matches=10)
    fb10 = recent_form(results_played, team_b, now, n_matches=10)

    form_diff = fa10["wr"] - fb10["wr"]

    # Win probability from ELO
    ep = elo_to_win_prob(elo_diff, neutral=neutral)

    # Composite strength
    sa = 0.35*(ea/2200) + 0.25*fa10["wr"]
    sb = 0.35*(eb/2200) + 0.25*fb10["wr"]

    # Build feature vector
    mf = {
        "elo_diff"              : elo_diff,
        "elo_win_prob"          : ep,
        "elo_diff_capped"       : np.clip(elo_diff, -200, 200),
        "elo_official_diff"     : elo_diff,
        "fifa_rank_diff"        : 0,
        "form5_win_rate_diff"   : fa5["wr"]  - fb5["wr"],
        "form10_win_rate_diff"  : form_diff,
        "form10_gd_diff"        : fa10["gd"] - fb10["gd"],
        "form10_gf_diff"        : fa10["gf"] - fb10["gf"],
        "avg_overall_diff"      : 0,
        "max_overall_diff"      : 0,
        "market_value_diff"     : 0,
        "h2h_win_rate_a"        : head_to_head_wr(results_played, team_a, team_b, now),
        "h2h_wc_win_rate_a"     : head_to_head_wr(results_played, team_a, team_b, now, wc_only=True),
        "shootout_win_rate_diff": get_shootout_wr(team_a) - get_shootout_wr(team_b),
        "avg_caps_diff"         : 0,
        "is_knockout"           : int(is_knockout),
        "neutral_venue"         : int(neutral),
        "strength_score_diff"   : sa - sb,
        "upset_potential"       : int(elo_diff < -50 and form_diff > 0.15),
        "ko_upset_risk"         : int(is_knockout) * (1 - ep),
        "form_elo_interaction"  : form_diff * (1 - ep),
    }

    x    = np.array([[mf.get(f, 0) for f in FEATURE_COLS]])
    x_sc = scaler.transform(x)

    # XGBoost probabilities
    proba   = xgb_model.predict_proba(x_sc)[0]
    p_away_xgb, p_draw_xgb, p_home_xgb = proba[0]*100, proba[1]*100, proba[2]*100

    # Poisson expected goals
    lam_h = max(0.3, poisson_home.predict(x_sc)[0])
    lam_a = max(0.3, poisson_away.predict(x_sc)[0])

    # Derive probabilities from Poisson score matrix
    ph = pd_ = pa = 0.0
    for h in range(10):
        for a in range(10):
            p = sp_poisson.pmf(h, lam_h) * sp_poisson.pmf(a, lam_a)
            if h > a:    ph  += p
            elif h == a: pd_ += p
            else:        pa  += p

    # Ensemble: average XGBoost and Poisson probabilities
    p_home = round((p_home_xgb + ph*100) / 2, 1)
    p_draw = round((p_draw_xgb + pd_*100) / 2, 1)
    p_away = round((p_away_xgb + pa*100)  / 2, 1)

    # Determine predicted winner
    if p_home >= p_away and p_home >= p_draw:
        winner = f"🏆 {team_a}"
    elif p_away >= p_home and p_away >= p_draw:
        winner = f"🏆 {team_b}"
    else:
        winner = "🤝 Draw"

    # Display results
    stage = "⚡ KNOCKOUT" if is_knockout else "🔵 Group Stage"
    upset = mf["upset_potential"]

    print(f"{'='*58}")
    print(f"  {team_a}  🆚  {team_b}")
    print(f"  {stage} | {'Neutral Venue' if neutral else 'Home Venue'}")
    print(f"{'='*58}")
    print(f"\n  Expected Score  :  {lam_h:.1f} — {lam_a:.1f}")
    print(f"\n  Win Probability (XGBoost + Poisson Ensemble):")
    print(f"    {team_a:<28} {p_home:>5.1f}%  {'█' * int(p_home/4)}")
    print(f"    {'Draw':<28} {p_draw:>5.1f}%  {'█' * int(p_draw/4)}")
    print(f"    {team_b:<28} {p_away:>5.1f}%  {'█' * int(p_away/4)}")
    print(f"\n  Prediction: {winner}")

    if upset:
        print(f"\n  ⚠️  UPSET RISK: {team_b} is weaker on ELO but in better recent form!")

    print(f"{'='*58}")

    return {
        "team_a": team_a, "team_b": team_b,
        "p_home": p_home, "p_draw": p_draw, "p_away": p_away,
        "exp_home": round(lam_h, 2), "exp_away": round(lam_a, 2),
        "winner": winner, "upset_risk": bool(upset),
    }

print("✅ Prediction function ready")


✅ Prediction function ready


<a id="16"></a>
## 16. 🎲 Monte Carlo Championship Simulation

Running 1,000 simulations of the remaining tournament to estimate each team's probability
of winning the 2026 World Cup. Every simulation randomly samples from the Poisson goal
distribution, with penalty shootouts decided by historical win rates.


In [14]:
# ============================================================
# MONTE CARLO CHAMPIONSHIP SIMULATION
# 1,000 tournament simulations for remaining teams
# ============================================================

def simulate_single_match(team_a, team_b):
    """
    Simulate one match using Poisson goal sampling.
    Ties in knockout matches are resolved using historical shootout win rates.
    """
    ea = elo_live.get(team_a, 1500)
    eb = elo_live.get(team_b, 1500)
    ed = ea - eb

    # Build minimal feature vector for speed
    ep = elo_to_win_prob(ed, neutral=True)
    mf_values = [ed, ep, np.clip(ed,-200,200), ed] + [0]*13 + [1, 1] + [ea/2200-eb/2200] + [0, (1-ep), 0]
    x    = np.array([mf_values[:len(FEATURE_COLS)]])
    x_sc = scaler.transform(x)

    lam_h = max(0.3, poisson_home.predict(x_sc)[0])
    lam_a = max(0.3, poisson_away.predict(x_sc)[0])

    goals_h = np.random.poisson(lam_h)
    goals_a = np.random.poisson(lam_a)

    if goals_h > goals_a:
        return team_a
    elif goals_a > goals_h:
        return team_b
    else:
        # Penalty shootout — decided by historical win rates
        so_a  = get_shootout_wr(team_a)
        so_b  = get_shootout_wr(team_b)
        total = so_a + so_b if (so_a + so_b) > 0 else 1
        return team_a if random.random() < so_a / total else team_b


def run_tournament_simulation(alive_teams, n_simulations=1000):
    """
    Simulate the full remaining tournament n times.
    Returns win probability for each team.
    """
    teams_list = sorted(alive_teams)
    wins  = {t: 0 for t in teams_list}
    finals = {t: 0 for t in teams_list}

    for sim in range(n_simulations):
        remaining = teams_list.copy()
        random.shuffle(remaining)
        round_num = 0

        while len(remaining) > 1:
            round_num += 1
            next_round = []
            for i in range(0, len(remaining) - 1, 2):
                winner = simulate_single_match(remaining[i], remaining[i+1])
                next_round.append(winner)
            if len(remaining) % 2 == 1:
                next_round.append(remaining[-1])  # Bye
            if len(remaining) <= 4:
                for t in next_round:
                    finals[t] += 1
            remaining = next_round

        wins[remaining[0]] += 1

    results_df = (
        pd.DataFrame([
            {"Team": t, "Wins": w, "Win %": round(w/n_simulations*100, 1),
             "Final %": round(finals.get(t,0)/n_simulations*100, 1)}
            for t, w in wins.items()
        ])
        .sort_values("Win %", ascending=False)
        .reset_index(drop=True)
    )
    results_df["#"] = results_df.index + 1
    return results_df


print(f"⏳ Running 1,000 Monte Carlo simulations on {len(still_alive)} remaining teams...")
mc_results = run_tournament_simulation(still_alive, n_simulations=1000)
mc_results.to_csv("monte_carlo_results.csv", index=False)

print(f"\n🏆 WC 2026 Championship Probabilities (1,000 Simulations):\n")
medals = ["🥇", "🥈", "🥉"]
for _, row in mc_results[mc_results["Win %"] > 0].iterrows():
    medal = medals[int(row["#"])-1] if int(row["#"]) <= 3 else f"{int(row['#'])}."
    bar = "█" * int(row["Win %"] / 2)
    print(f"  {medal} {row['Team']:<25} {row['Win %']:>5.1f}%  {bar}")

print(f"\n✅ Results saved to: monte_carlo_results.csv")


⏳ Running 1,000 Monte Carlo simulations on 4 remaining teams...

🏆 WC 2026 Championship Probabilities (1,000 Simulations):

  🥇 Spain                      41.4%  ████████████████████
  🥈 Argentina                  38.0%  ███████████████████
  🥉 Colombia                   13.9%  ██████
  4. Australia                   6.7%  ███

✅ Results saved to: monte_carlo_results.csv


<a id="17"></a>
## 17. 🔮 Results & Sample Predictions


In [15]:
# ============================================================
# SAMPLE MATCH PREDICTIONS — Current R32 Matches
# ============================================================

print("🔮 Predicting upcoming Round of 32 matches...\n")

upcoming_fixtures = [
    ("Spain",         "Austria",        True),
    ("Portugal",      "Croatia",        True),
    ("Switzerland",   "Algeria",        True),
    ("Argentina",     "Cape Verde",     True),
    ("Australia",     "Egypt",          True),
    ("Colombia",      "Ghana",          True),
    ("Canada",        "Morocco",        True),
    ("Paraguay",      "France",         True),
    ("Mexico",        "England",        True),
    ("Brazil",        "Norway",         True),
    ("United States", "Belgium",        True),
]

all_predictions = []
for team_a, team_b, is_ko in upcoming_fixtures:
    print(f"\n{'-'*58}")
    r = predict_match(team_a, team_b, is_knockout=is_ko, neutral=True)
    all_predictions.append(r)

# Summary table
print(f"\n\n{'='*70}")
print(f"  PREDICTIONS SUMMARY")
print(f"{'='*70}")
print(f"  {'Match':<35} {'Expected':<12} {'Winner'}")
print(f"{'-'*70}")
for p in all_predictions:
    match  = f"{p['team_a']} vs {p['team_b']}"
    score  = f"{p['exp_home']:.1f} — {p['exp_away']:.1f}"
    winner = p['winner']
    print(f"  {match:<35} {score:<12} {winner}")


🔮 Predicting upcoming Round of 32 matches...


----------------------------------------------------------
  Spain  🆚  Austria
  ⚡ KNOCKOUT | Neutral Venue

  Expected Score  :  2.6 — 0.7

  Win Probability (XGBoost + Poisson Ensemble):
    Spain                         82.7%  ████████████████████
    Draw                           9.3%  ██
    Austria                        8.0%  ██

  Prediction: 🏆 Spain

----------------------------------------------------------
  Portugal  🆚  Croatia
  ⚡ KNOCKOUT | Neutral Venue

  Expected Score  :  1.8 — 0.9

  Win Probability (XGBoost + Poisson Ensemble):
    Portugal                      54.1%  █████████████
    Draw                          28.8%  ███████
    Croatia                       17.1%  ████

  Prediction: 🏆 Portugal

----------------------------------------------------------
  Switzerland  🆚  Algeria
  ⚡ KNOCKOUT | Neutral Venue

  Expected Score  :  2.2 — 0.8

  Win Probability (XGBoost + Poisson Ensemble):
    Switzerland           

<a id="18"></a>
## 18. 🖥️ Interactive Dashboard

This notebook is accompanied by a **full Streamlit web application** deployed live.

### Dashboard Features (6 Pages)

| Page | Description |
|------|-------------|
| 🏠 **Home** | Live tournament stats · ELO leaderboard · All 12 groups |
| ⚔️ **Predict Match** | Any two teams · Win/Draw/Loss probabilities · Detailed comparison |
| 🔵 **Group Stage** | Monte Carlo standings simulation for any group |
| 📅 **Next Round** | All upcoming R32 matches with predictions |
| 🏆 **Tournament Winner** | Monte Carlo championship probability per team |
| 📄 **PDF Report** | Downloadable full prediction report |

### Extra Features
- 🌙 Dark / ☀️ Light mode toggle
- 🇸🇦 Arabic / 🇬🇧 English language support
- 🔄 One-click live refresh from GitHub

---

### Links

| | |
|---|---|
| 🔴 **Live App** | https://wc2026-match-predictor-worldcup.streamlit.app/ |
| 🐙 **GitHub** | https://github.com/tito644/wc2026-match-predictor |

---

## 👤 Author

**Tariq Elnaggar** — Data Scientist

| | |
|---|---|
| 🐙 GitHub | https://github.com/tito644 |
| 💼 LinkedIn | https://www.linkedin.com/in/tarek-mohamed-el-naggar/ |
| 🏆 Kaggle | https://www.kaggle.com/tarekelnaggar |

---

<div align="center">

*"Football is unpredictable. That's exactly what makes predicting it so fascinating."*

**⭐ If you found this useful, please upvote — it helps others discover the work!**

</div>
